# Data exploration notebook
This is the starting notebook of the project: its goal is to understand the structure
of the data before using it for any analysis.
All data comes from [FastF1](https://docs.fastf1.dev/), a Python library that gives access
to Formula 1 timing, tyre and telemetry data.

## Setup
Imports and FastF1 cache configuration: downloaded data is stored in the `cache/`
folder at the repo root, so each session is only downloaded once.

In [ ]:
import fastf1
import pandas as pd
import datetime

from f1analysis.data import enable_cache

enable_cache()
pd.set_option("display.max_columns", None)

## Available seasons

According to the documentation, timing data, session
information, car telemetry and position data are available from 2018 onwards.

`get_event_schedule(year)` is the library's top-level entry point: it only needs a year
and returns that season's calendar. Let's start by loading every season from 2018 to the current one
and counting how many events each contains.

In [21]:
current_year = datetime.date.today().year
years = range(2018, current_year + 1)

schedules = {year: fastf1.get_event_schedule(year) for year in years}

pd.Series({year: len(schedule) for year, schedule in schedules.items()}, name="Events")

2018    21
2019    21
2020    19
2021    23
2022    24
2023    23
2024    25
2025    25
2026    25
Name: Events, dtype: int64

These counts are higher than the actual number of Grands Prix in some seasons
(e.g. 2020 had 17 races, not 19). By default `get_event_schedule` also returns
pre-season testing events (`include_testing=True`), which have `RoundNumber == 0`.
Let's check whether they explain the difference by counting Grands Prix and testing
events separately.

In [22]:
pd.DataFrame({
    "GPs": {year: (s["RoundNumber"] > 0).sum() for year, s in schedules.items()},
    "Testing":     {year: (s["RoundNumber"] == 0).sum() for year, s in schedules.items()},
})

,GPs,Testing
2018,21,0
2019,21,0
2020,17,2
2021,22,1
2022,22,2
2023,22,1
2024,24,1
2025,24,1
2026,23,2


Testing events explain the difference: once they are removed, the counts match the
real number of Grands Prix, from 17 (2020) to 24 (2024–2025). FastF1 lists testing
only from 2020 onwards.

Since testing is not part of the championship, it is excluded from now on.

In [26]:
schedules = {year: s[s["RoundNumber"] > 0] for year, s in schedules.items()}

## Season calendar

With the available seasons defined, the next question is what a single season calendar
contains: the 2025 season is used as an example, since it is the most recent complete one.

In [30]:
season = schedules[2025]
season

,RoundNumber,Country,Location,OfficialEventName,EventDate,EventName,EventFormat,Session1,Session1Date,Session1DateUtc,Session2,Session2Date,Session2DateUtc,Session3,Session3Date,Session3DateUtc,Session4,Session4Date,Session4DateUtc,Session5,Session5Date,Session5DateUtc,F1ApiSupport
1,1,Australia,Melbourne,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Australian Grand Prix,conventional,Practice 1,2025-03-14 12:30:00+11:00,2025-03-14 01:30:00,Practice 2,2025-03-14 16:00:00+11:00,2025-03-14 05:00:00,Practice 3,2025-03-15 12:30:00+11:00,2025-03-15 01:30:00,Qualifying,2025-03-15 16:00:00+11:00,2025-03-15 05:00:00,Race,2025-03-16 15:00:00+11:00,2025-03-16 04:00:00,True
2,2,China,Shanghai,FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2025,2025-03-23,Chinese Grand Prix,sprint_qualifying,Practice 1,2025-03-21 11:30:00+08:00,2025-03-21 03:30:00,Sprint Qualifying,2025-03-21 15:30:00+08:00,2025-03-21 07:30:00,Sprint,2025-03-22 11:00:00+08:00,2025-03-22 03:00:00,Qualifying,2025-03-22 15:00:00+08:00,2025-03-22 07:00:00,Race,2025-03-23 15:00:00+08:00,2025-03-23 07:00:00,True
3,3,Japan,Suzuka,FORMULA 1 LENOVO JAPANESE GRAND PRIX 2025,2025-04-06,Japanese Grand Prix,conventional,Practice 1,2025-04-04 11:30:00+09:00,2025-04-04 02:30:00,Practice 2,2025-04-04 15:00:00+09:00,2025-04-04 06:00:00,Practice 3,2025-04-05 11:30:00+09:00,2025-04-05 02:30:00,Qualifying,2025-04-05 15:00:00+09:00,2025-04-05 06:00:00,Race,2025-04-06 14:00:00+09:00,2025-04-06 05:00:00,True
4,4,Bahrain,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2025,2025-04-13,Bahrain Grand Prix,conventional,Practice 1,2025-04-11 14:30:00+03:00,2025-04-11 11:30:00,Practice 2,2025-04-11 18:00:00+03:00,2025-04-11 15:00:00,Practice 3,2025-04-12 15:30:00+03:00,2025-04-12 12:30:00,Qualifying,2025-04-12 19:00:00+03:00,2025-04-12 16:00:00,Race,2025-04-13 18:00:00+03:00,2025-04-13 15:00:00,True
5,5,Saudi Arabia,Jeddah,FORMULA 1 STC SAUDI ARABIAN GRAND PRIX 2025,2025-04-20,Saudi Arabian Grand Prix,conventional,Practice 1,2025-04-18 16:30:00+03:00,2025-04-18 13:30:00,Practice 2,2025-04-18 20:00:00+03:00,2025-04-18 17:00:00,Practice 3,2025-04-19 16:30:00+03:00,2025-04-19 13:30:00,Qualifying,2025-04-19 20:00:00+03:00,2025-04-19 17:00:00,Race,2025-04-20 20:00:00+03:00,2025-04-20 17:00:00,True
6,6,United States,Miami Gardens,FORMULA 1 CRYPTO.COM MIAMI GRAND PRIX 2025,2025-05-04,Miami Grand Prix,sprint_qualifying,Practice 1,2025-05-02 12:30:00-04:00,2025-05-02 16:30:00,Sprint Qualifying,2025-05-02 16:30:00-04:00,2025-05-02 20:30:00,Sprint,2025-05-03 12:00:00-04:00,2025-05-03 16:00:00,Qualifying,2025-05-03 16:00:00-04:00,2025-05-03 20:00:00,Race,2025-05-04 16:00:00-04:00,2025-05-04 20:00:00,True
7,7,Italy,Imola,FORMULA 1 AWS GRAN PREMIO DEL MADE IN ITALY E ...,2025-05-18,Emilia Romagna Grand Prix,conventional,Practice 1,2025-05-16 13:30:00+02:00,2025-05-16 11:30:00,Practice 2,2025-05-16 17:00:00+02:00,2025-05-16 15:00:00,Practice 3,2025-05-17 12:30:00+02:00,2025-05-17 10:30:00,Qualifying,2025-05-17 16:00:00+02:00,2025-05-17 14:00:00,Race,2025-05-18 15:00:00+02:00,2025-05-18 13:00:00,True
8,8,Monaco,Monaco,FORMULA 1 TAG HEUER GRAND PRIX DE MONACO 2025,2025-05-25,Monaco Grand Prix,conventional,Practice 1,2025-05-23 13:30:00+02:00,2025-05-23 11:30:00,Practice 2,2025-05-23 17:00:00+02:00,2025-05-23 15:00:00,Practice 3,2025-05-24 12:30:00+02:00,2025-05-24 10:30:00,Qualifying,2025-05-24 16:00:00+02:00,2025-05-24 14:00:00,Race,2025-05-25 15:00:00+02:00,2025-05-25 13:00:00,True
9,9,Spain,Barcelona,FORMULA 1 ARAMCO GRAN PREMIO DE ESPAÑA 2025,2025-06-01,Spanish Grand Prix,conventional,Practice 1,2025-05-30 13:30:00+02:00,2025-05-30 11:30:00,Practice 2,2025-05-30 17:00:00+02:00,2025-05-30 15:00:00,Practice 3,2025-05-31 12:30:00+02:00,2025-05-31 10:30:00,Qualifying,2025-05-31 16:00:00+02:00,2025-05-31 14:00:00,Race,2025-06-01 15:00:00+02:00,2025-06-01 13:00:00,True
10,10,Canada,Montréal,FORMULA 1 PIRELLI GRAND PRIX DU CANADA 2025,2025-06-15,Canadian Grand Prix,conventional,Practice 1,2025-06-13 13:30:00-04:00,202

Each row represents a Grand Prix. The columns of the calendar can be grouped as follows:

- **Identity**: `RoundNumber`, `EventName`, `OfficialEventName`
- **Location**: `Country`, `Location`
- **Date**: `EventDate`
- **Format**: `EventFormat`
- **Sessions**: `Session1` to `Session5`, each with its local and UTC date
- **Data availability**: `F1ApiSupport`

Each group is explored below, starting from the identity and location of each event.

In [31]:
season[["RoundNumber", "EventName", "OfficialEventName", "Country", "Location"]]

,RoundNumber,EventName,OfficialEventName,Country,Location
1,1,Australian Grand Prix,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,Australia,Melbourne
2,2,Chinese Grand Prix,FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2025,China,Shanghai
3,3,Japanese Grand Prix,FORMULA 1 LENOVO JAPANESE GRAND PRIX 2025,Japan,Suzuka
4,4,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2025,Bahrain,Sakhir
5,5,Saudi Arabian Grand Prix,FORMULA 1 STC SAUDI ARABIAN GRAND PRIX 2025,Saudi Arabia,Jeddah
6,6,Miami Grand Prix,FORMULA 1 CRYPTO.COM MIAMI GRAND PRIX 2025,United States,Miami Gardens
7,7,Emilia Romagna Grand Prix,FORMULA 1 AWS GRAN PREMIO DEL MADE IN ITALY E ...,Italy,Imola
8,8,Monaco Grand Prix,FORMULA 1 TAG HEUER GRAND PRIX DE MONACO 2025,Monaco,Monaco
9,9,Spanish Grand Prix,FORMULA 1 ARAMCO GRAN PREMIO DE ESPAÑA 2025,Spain,Barcelona
10,10,Canadian Grand Prix,FORMULA 1 PIRELLI GRAND PRIX DU CANADA 2025,Canada,Montréal


- `EventName` is the clearest way to identify a Grand Prix: short and consistent across seasons.
- `OfficialEventName` includes the title sponsor, so it changes over the years and is not useful for analysis.
- `Location` is the name of the circuit's town, often the name used in everyday language
  (e.g. Monza, Imola).
- `RoundNumber` is the position of the event in the season.

Together, year and `RoundNumber` identify an event without ambiguity. Names and locations
might not: the next cell checks whether a location can host more than one event in the same season.

In [32]:
for year, s in schedules.items():
    repeated = s[s["Location"].duplicated(keep=False)]
    if not repeated.empty:
        print(year, repeated[["RoundNumber", "EventName", "Location"]].to_dict("records"))

2020 [{'RoundNumber': 1, 'EventName': 'Austrian Grand Prix', 'Location': 'Spielberg'}, {'RoundNumber': 2, 'EventName': 'Styrian Grand Prix', 'Location': 'Spielberg'}, {'RoundNumber': 4, 'EventName': 'British Grand Prix', 'Location': 'Silverstone'}, {'RoundNumber': 5, 'EventName': '70th Anniversary Grand Prix', 'Location': 'Silverstone'}, {'RoundNumber': 15, 'EventName': 'Bahrain Grand Prix', 'Location': 'Sakhir'}, {'RoundNumber': 16, 'EventName': 'Sakhir Grand Prix', 'Location': 'Sakhir'}]
2021 [{'RoundNumber': 8, 'EventName': 'Styrian Grand Prix', 'Location': 'Spielberg'}, {'RoundNumber': 9, 'EventName': 'Austrian Grand Prix', 'Location': 'Spielberg'}]
